In [2]:
"""
Langchain agent of SQL Assistant.
"""
import logging
import os
import time

import pyodbc
from dynaconf import Dynaconf
from typing import Optional, Dict, Any
# from langchain.agents.format_scratchpad import format_to_tool_messages
from langchain.agents.output_parsers.tools import ToolAgentAction, parse_ai_message_to_tool_action
from langchain_groq import ChatGroq
from langchain.tools import tool

# from langchain.agents import AgentExecutor
from langchain_core.agents import AgentFinish
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents.format_scratchpad.openai_tools import (format_to_openai_tool_messages)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser
from pydantic import BaseModel, Field

from data_models import MessagesList, Message, Sender


### Tools

In [19]:

class ExecuteSQLQueryInput(BaseModel):
    query: str = Field(description="The SQL query to be executed on the relational database.")
    limit: Optional[int] = Field(
        default=5,
        description="Maximum number of rows to return from the result set."
    )


@tool("execute_sql_query", args_schema=ExecuteSQLQueryInput, return_direct=False)
def execute_sql_query(query: str, limit: int = 5) -> str:
    """
    Executes an SQL query on the relational database and returns the results.
    The results are returned in a structured JSON format including column names and row values.
    """
    pass


def execute_sql_query_imp(settings: Dynaconf, logger: logging.Logger,
                          conn_str: str,
                          query: str, limit: int = 5) -> Dict[str, Any]:
    """
    Execute an SQL query and return structured results.
    - If query fails: return structured error.
    - If results exceed `limit`: return only first `limit` rows and note clipping.
    """

    try:
        if limit is None:
            limit = 20  # default cap if not provided
        else:
            limit = min(limit, 20)  # hard cap at 10 rows
        if limit is not None and query.startswith('"') and query.endswith('"'):
            query = query[1:-1]

        cnxn = pyodbc.connect(conn_str, autocommit=False)
        cursor = cnxn.cursor()

        cursor.execute(query)

        # If query has no result set (like DDL/DML), just return success message
        if cursor.description is None:
            return {
                "status": "success",
                "message": "✅ Query executed successfully. No rows returned.",
                "rows_returned": 0,
                "data": []
            }

        # Extract column names
        columns = [col[0] for col in cursor.description]

        # Fetch rows
        rows = cursor.fetchall()
        total_rows = len(rows)

        # Clip rows if needed
        clipped = False
        if total_rows > limit:
            rows = rows[:limit]
            clipped = True

        # Convert rows into list of dicts
        data = [dict(zip(columns, row)) for row in rows]

        result = {
            "status": "success",
            "message": "✅ Query executed successfully.",
            "rows_returned": total_rows,
            "data": data,
        }

        if clipped:
            result["note"] = f"⚠️ {total_rows} rows received. Showing first {limit} rows only."

        return result

    except Exception as e:
        logger.error(f"SQL Execution Error: {e}")
        return {
            "status": "error",
            "message": f"❌ Query failed with error: {str(e)}",
            "rows_returned": 0,
            "data": []
        }


In [20]:
# #### Testing Code ####
from utils import get_settings, get_logger, get_db_connection
settings = get_settings()
logger = get_logger(settings)
db_conn_str: str = get_db_connection(settings, logger)

In [21]:
def get_tables_metadata(settings: Dynaconf):
    with open(settings.get("METADATA_PATH"), "r", encoding="utf-8") as file:
        metadata = file.read()
    return metadata
metadata = get_tables_metadata(settings)

### Agent

In [1]:
def sql_agent(settings: Dynaconf, logger: logging.getLogger, db_conn_str: str, conversation: MessagesList) -> str:
    try:
        
        global metadata
        os.environ["OPENAI_API_KEY"] = settings.get("OPENAI_API_KEY")
        tier = "priority" if settings.current_env == "PRODUCTION" else "default"
        # tier = "priority"
        # model = "gpt-5-mini"  # "gpt-4o" "gpt-3.5-turbo"  "gpt-4-0125-preview"
        # llm = ChatOpenAI(model=model, temperature=1, service_tier=tier)  # temperature=1 for gpt-5, for others you can change
        os.environ["GROQ_API_KEY"] = settings.get("GROQ_API_KEY")
        llm = ChatGroq(model="meta-llama/llama-4-maverick-17b-128e-instruct", temperature=0)
        # llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

        tools = [
            execute_sql_query
        ]

        system_prompt = """"""

        user_prompt = """
Conversation:
{conversation}
        """

        agent_prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system_prompt),
                ("user", user_prompt),
                MessagesPlaceholder(variable_name="agent_scratchpad"),
            ]
        )

        llm_with_tools = llm.bind_tools(tools)

        agent = (
                {
                    "conversation": lambda x: x["conversation"],
                    "metadata": lambda x: x["metadata"],
                    "agent_scratchpad": lambda x: format_to_openai_tool_messages(
                        x["intermediate_steps"]
                    ),
                }
                | agent_prompt
                | llm_with_tools
                | OpenAIToolsAgentOutputParser()
            # OpenAIToolsAgentOutputParser()  # PydanticToolsParser(tools=[tools list...])
            # parser has applied parse_ai_message_to_tool_action method
        )

        prompt_input = {
            "conversation": conversation,
            "metadata": metadata,
            "intermediate_steps": []
        }
        # agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
        # output = agent_executor.invoke(prompt_input)
        # logger.info(output)

        testing_prompt = agent_prompt.format(**{
            "conversation": prompt_input["conversation"],
            "metadata": prompt_input["metadata"],
            "agent_scratchpad": format_to_openai_tool_messages(prompt_input["intermediate_steps"])
        })
        agent_invoke_start_time = time.time()
        output = agent.invoke(prompt_input)
        logger.info(f"agent.invoke took {time.time() - agent_invoke_start_time:.2f} seconds")
        logger.info(output)

        run = True
        while not isinstance(output, AgentFinish) and run:
            for selected_tool in output:
                logger.info("Tool call:     --->    " + selected_tool.log.strip())
                if isinstance(selected_tool, ToolAgentAction):
                    name = selected_tool.tool
                    tool_input = selected_tool.tool_input

                    if name == "execute_sql_query":
                        start_time = time.time()
                        tool_output = execute_sql_query_imp(settings, logger, db_conn_str, **tool_input)
                        elapsed_time = time.time() - start_time
                        logger.info(f"execute_sql_query_imp took {elapsed_time:.2f} seconds")

                    logger.info("Observation:   --->    " + str(tool_output))
                    prompt_input["intermediate_steps"].append((
                        # AgentAction(tool=name, tool_input=tool_input, log=selected_tool.log),
                        selected_tool, tool_output
                    ))

            testing_prompt = agent_prompt.format(**{
                "conversation": prompt_input["conversation"],
                "metadata": prompt_input["metadata"],
                "agent_scratchpad": format_to_openai_tool_messages(prompt_input["intermediate_steps"])
            })
            agent_invoke_start_time = time.time()
            output = agent.invoke(prompt_input)
            logger.info(f"agent.invoke took {time.time() - agent_invoke_start_time:.2f} seconds")
            logger.info(output)

            # run = False
        output = output.messages[0].content
        # logger.info("Final output:   --->    " + output)
        return output
    except Exception as e:
        logger.warning(f"Error in sql agent:\n{(str(e))}")
        return f"Error in sql agent:\n{str(e)}"


# # # #### Tools testing ####
# output = execute_sql_query_imp(settings, logger, db_conn_str,"SELECT * FROM Products", limit=3)
# print(output)
# #
# Sample conversation
conversation = MessagesList()
# Adding messages to the conversation
conversation.add_message(Message(text="What is the total sale?", sender=Sender.USER))
# conversation.add_message(Message(text="Sure, I'm here to help. What seems to be the problem?", sender=Sender.ASSISTANT))
print(f"Agent Output:\n{sql_agent(settings, logger, db_conn_str, conversation)}")

NameError: name 'Dynaconf' is not defined